In [1]:
import requests
from bs4 import BeautifulSoup, Tag
import csv
import json
import re

URL = "https://www.osha.gov/confined-spaces-construction/faq"

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text

def scrape_osha_faq(url: str):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    faqs = []
    current_q = None
    answer_parts = []

    # OSHA page uses headings like: #### 1. What is a confined space?
    # In HTML these are usually h4 elements.
    content_root = soup.find("main") or soup.body

    for el in content_root.descendants:
        if not isinstance(el, Tag):
            continue

        # Skip nested tags unless they are block-ish content or headings
        if el.name not in {"h4", "p", "ul", "ol", "li", "div"}:
            continue

        text = clean_text(el.get_text(" ", strip=True))
        if not text:
            continue

        # Detect question headings
        if el.name == "h4" and re.match(r"^\d+\.\s", text):
            # save previous FAQ
            if current_q is not None:
                faqs.append({
                    "question_number": current_q["question_number"],
                    "question": current_q["question"],
                    "answer": clean_text(" ".join(answer_parts))
                })

            m = re.match(r"^(\d+)\.\s*(.+)$", text)
            current_q = {
                "question_number": int(m.group(1)),
                "question": m.group(2).strip()
            }
            answer_parts = []
            continue

        # collect answer text until next question
        if current_q is not None:
            # ignore decorative separators / footer junk
            if text in {"* * *", "Scroll to Top"}:
                continue
            answer_parts.append(text)

    # save last one
    if current_q is not None:
        faqs.append({
            "question_number": current_q["question_number"],
            "question": current_q["question"],
            "answer": clean_text(" ".join(answer_parts))
        })

    # optional cleanup: remove obvious footer bleed if any
    cleaned = []
    for item in faqs:
        answer = item["answer"]

        # stop at footer markers if they accidentally got included
        footer_markers = [
            "U.S. Department of Labor",
            "Occupational Safety and Health Administration",
            "Federal Government"
        ]
        for marker in footer_markers:
            if marker in answer:
                answer = answer.split(marker)[0].strip()

        cleaned.append({
            "question_number": item["question_number"],
            "question": item["question"],
            "answer": answer
        })

    return cleaned


if __name__ == "__main__":
    faqs = scrape_osha_faq(URL)

    # Save JSON
    with open("osha_confined_spaces_faq.json", "w", encoding="utf-8") as f:
        json.dump(faqs, f, indent=2, ensure_ascii=False)

    # Save CSV
    with open("osha_confined_spaces_faq.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["question_number", "question", "answer"])
        writer.writeheader()
        writer.writerows(faqs)

    print(f"Scraped {len(faqs)} FAQs")
    print("Saved to:")
    print("- osha_confined_spaces_faq.json")
    print("- osha_confined_spaces_faq.csv")

Scraped 72 FAQs
Saved to:
- osha_confined_spaces_faq.json
- osha_confined_spaces_faq.csv
